# Attaching Pre-Existing Trial Data with Ax's Client

In many real-world scenarios you may already have data from previous experiments — perhaps logged in a spreadsheet, database, or DataFrame — and you want to leverage Ax's modeling and analysis capabilities on top of that data before generating new candidates.

This tutorial demonstrates how to:
1. Create an experiment using Ax's `Client`
2. Attach historical trials with arms (and their parameters) from a DataFrame
3. Attach metric observations from a DataFrame
4. Analyze the results using Ax's built-in analyses (cross-validation, utility progression, and arm effects)
5. Generate a new candidate trial using Ax's Bayesian optimization model
6. Visualize the predicted effects of the new candidate

### When is this useful?
- You ran experiments outside of Ax and want to use Ax to model them and suggest next steps
- You have a CSV or database table of past configurations and their outcomes
- You want to warm-start Bayesian optimization with historical data

### Prerequisites
- Familiarity with Python and pandas DataFrames
- Basic understanding of [adaptive experimentation](../../intro-to-ae.mdx) and [Bayesian optimization](../../intro-to-bo.mdx)

## Step 1: Import Necessary Modules

In [ ]:
import numpy as np
import pandas as pd
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig

## Step 2: Prepare Historical Data

Imagine we've previously evaluated the [Branin function](https://www.sfu.ca/~ssurjano/branin.html) at several points and recorded the results in a DataFrame. The Branin function is a common benchmark for optimization with two parameters ($x_1$ and $x_2$) and a known global minimum of $\approx 0.398$.

$$
f(x_1, x_2) = \left(x_2 - \frac{5.1}{4\pi^2} x_1^2 + \frac{5}{\pi} x_1 - 6\right)^2 + 10 \left(1 - \frac{1}{8\pi}\right) \cos(x_1) + 10
$$

We'll create a DataFrame representing this historical data — each row is a previously-evaluated configuration with its observed metric value.

In [3]:
def branin(x1: float, x2: float) -> float:
    """The Branin function — a standard optimization benchmark."""
    return (
        (x2 - 5.1 / (4 * np.pi**2) * x1**2 + 5.0 / np.pi * x1 - 6.0) ** 2
        + 10 * (1 - 1.0 / (8 * np.pi)) * np.cos(x1)
        + 10
    )


# Generate historical data: 15 quasi-random evaluations
rng = np.random.default_rng(42)
n_historical = 15
historical_data = pd.DataFrame(
    {
        "arm_name": [f"historical_{i}" for i in range(n_historical)],
        "x1": rng.uniform(-5, 10, n_historical),
        "x2": rng.uniform(0, 15, n_historical),
    }
)
historical_data["branin"] = historical_data.apply(
    lambda row: branin(row["x1"], row["x2"]), axis=1
)

historical_data

,arm_name,x1,x2,branin
0,historical_0,6.609341,3.408581,24.314643
1,historical_1,1.583177,8.318772,30.263474
2,historical_2,7.878969,0.957259,10.033110
3,historical_3,5.460520,12.414468,143.167098
4,historical_4,-3.587340,9.474966,16.522254
5,historical_5,9.634335,11.371316,76.539605
6,historical_6,6.417096,5.317890,37.251285
7,historical_7,6.790965,14.560470,198.245457
8,historical_8,-3.078296,13.396817,2.038708
9,historical_9,1.755789,11.675752,73.389817


## Step 3: Initialize the Client and Configure the Experiment

We create a `Client` and define the search space to match our historical data. The Branin function is typically evaluated on $x_1 \in [-5, 10]$ and $x_2 \in [0, 15]$.

In [4]:
client = Client(random_seed=42)

client.configure_experiment(
    name="branin_historical",
    parameters=[
        RangeParameterConfig(name="x1", parameter_type="float", bounds=(-5.0, 10.0)),
        RangeParameterConfig(name="x2", parameter_type="float", bounds=(0.0, 15.0)),
    ],
)

client.configure_optimization(objective="-branin")  # minimize
client.configure_generation_strategy()

[INFO 02-19 16:20:55] ax.api.client: GenerationStrategy(name='Center+Sobol+MBM:fast', nodes=[CenterGenerationNode(next_node_name='Sobol', use_existing_trials_for_initialization=True), GenerationNode(name='Sobol', generator_specs=[GeneratorSpec(generator_enum=Sobol, generator_key_override=None)], transition_criteria=[MinTrials(transition_to='MBM'), MinTrials(transition_to='MBM')], suggested_experiment_status=ExperimentStatus.INITIALIZATION), GenerationNode(name='MBM', generator_specs=[GeneratorSpec(generator_enum=BoTorch, generator_key_override=None)], transition_criteria=[], suggested_experiment_status=ExperimentStatus.OPTIMIZATION)]) chosen based on user input and problem structure.


## Step 4: Attach Historical Trials from the DataFrame

Now we iterate over our DataFrame to attach each historical evaluation as a trial. For each row we:
1. Call `client.attach_trial` with the arm's parameters and name — this creates a trial in the RUNNING state
2. Call `client.complete_trial` with the observed metric data — this marks the trial COMPLETED

The `raw_data` argument to `complete_trial` is a dictionary mapping metric names to their observed values. Each value can be either a `float` (mean only) or a `tuple[float, float]` (mean, SEM) if you have uncertainty estimates.

In [5]:
for _, row in historical_data.iterrows():
    # Attach the trial with its parameterization
    trial_index = client.attach_trial(
        parameters={"x1": row["x1"], "x2": row["x2"]},
        arm_name=row["arm_name"],
    )

    # Complete the trial with observed metric data
    client.complete_trial(
        trial_index=trial_index,
        raw_data={"branin": row["branin"]},
    )

print(f"Attached {len(historical_data)} historical trials.")

[INFO 02-19 16:20:55] ax.api.client: Trial 0 marked COMPLETED.
[INFO 02-19 16:20:55] ax.api.client: Trial 1 marked COMPLETED.
[INFO 02-19 16:20:55] ax.api.client: Trial 2 marked COMPLETED.
[INFO 02-19 16:20:55] ax.api.client: Trial 3 marked COMPLETED.
[INFO 02-19 16:20:55] ax.api.client: Trial 4 marked COMPLETED.
[INFO 02-19 16:20:55] ax.api.client: Trial 5 marked COMPLETED.
[INFO 02-19 16:20:55] ax.api.client: Trial 6 marked COMPLETED.
[INFO 02-19 16:20:55] ax.api.client: Trial 7 marked COMPLETED.
[INFO 02-19 16:20:55] ax.api.client: Trial 8 marked COMPLETED.
[INFO 02-19 16:20:55] ax.api.client: Trial 9 marked COMPLETED.
[INFO 02-19 16:20:55] ax.api.client: Trial 10 marked COMPLETED.
[INFO 02-19 16:20:55] ax.api.client: Trial 11 marked COMPLETED.
[INFO 02-19 16:20:55] ax.api.client: Trial 12 marked COMPLETED.
[INFO 02-19 16:20:55] ax.api.client: Trial 13 marked COMPLETED.
[INFO 02-19 16:20:55] ax.api.client: Trial 14 marked COMPLETED.


Attached 15 historical trials.


We can inspect the state of the experiment using `client.summarize()`, which returns a DataFrame with one row per arm showing its parameters, metric values, and trial status.

In [6]:
client.summarize()

,trial_index,arm_name,trial_status,branin,x1,x2
0,0,historical_0,COMPLETED,24.314643,6.609341,3.408581
1,1,historical_1,COMPLETED,30.263474,1.583177,8.318772
2,2,historical_2,COMPLETED,10.033110,7.878969,0.957259
3,3,historical_3,COMPLETED,143.167098,5.460520,12.414468
4,4,historical_4,COMPLETED,16.522254,-3.587340,9.474966
5,5,historical_5,COMPLETED,76.539605,9.634335,11.371316
6,6,historical_6,COMPLETED,37.251285,6.417096,5.317890
7,7,historical_7,COMPLETED,198.245457,6.790965,14.560470
8,8,historical_8,COMPLETED,2.038708,-3.078296,13.396817
9,9,historical_9,COMPLETED,73.389817,1.755789,11.675752


## Step 5: Analyze the Experiment

With historical data attached, we can use Ax's analysis framework to understand the experiment so far. We'll compute three analyses:

1. **Cross-Validation Plot**: Assesses how well Ax's surrogate model fits the observed data using leave-one-out cross-validation. Points near the diagonal $y = x$ line indicate good model fit.

2. **Utility Progression**: Shows how the best observed objective value improves over completed trials. This helps us understand whether the historical evaluations were trending toward the optimum.

3. **Arm Effects Plot (Observed)**: Displays the observed metric values for each arm with confidence intervals, providing an overview of all evaluations.

In [ ]:
from ax.analysis.plotly.cross_validation import CrossValidationPlot
from ax.analysis.plotly.utility_progression import UtilityProgressionAnalysis
from ax.analysis.plotly.arm_effects import ArmEffectsPlot
cards = client.compute_analyses(
    analyses=[
        CrossValidationPlot(),
        UtilityProgressionAnalysis(),
        ArmEffectsPlot(metric_name="branin", use_model_predictions=False),
    ],
    display=True,
)

## Step 6: Generate a New Candidate Trial

Now that Ax has a trained model based on the historical data, we can ask it to suggest the next best point to evaluate. Ax uses Bayesian optimization to balance exploration (trying new regions) with exploitation (refining promising regions).

In [8]:
new_trials = client.get_next_trials(max_trials=1)

for trial_index, parameters in new_trials.items():
    print(f"Trial {trial_index}: {parameters}")

[INFO 02-19 16:21:07] ax.api.client: Generated new trial 15 with parameters {'x1': -5.0, 'x2': 15.0} using GenerationNode MBM.


Trial 15: {'x1': -5.0, 'x2': 15.0}


## Step 7: Visualize Predicted Arm Effects for the Candidate

Before actually evaluating the candidate, we can use the `ArmEffectsPlot` with `use_model_predictions=True` to see what the model predicts for all arms — including the newly generated candidate. This is useful for understanding:
- How the model expects the new candidate to perform relative to historical arms
- The model's uncertainty about each arm's true effect
- Whether the candidate is expected to improve upon the best historical result

The candidate trial will appear highlighted with a distinct color.

In [ ]:
cards = client.compute_analyses(
    analyses=[
        ArmEffectsPlot(metric_name="branin", use_model_predictions=True),
    ],
    display=True,
)

## Step 8: Evaluate the Candidate and Continue the Loop

In practice you would evaluate the candidate in your system, then report the result back to Ax and continue the optimization loop. Here we evaluate the Branin function directly and complete the trial.

In [10]:
for trial_index, parameters in new_trials.items():
    result = branin(parameters["x1"], parameters["x2"])
    client.complete_trial(
        trial_index=trial_index,
        raw_data={"branin": result},
    )
    print(f"Trial {trial_index} completed with branin = {result:.4f}")

[INFO 02-19 16:21:11] ax.api.client: Trial 15 marked COMPLETED.


Trial 15 completed with branin = 17.5083


Let's run a few more optimization rounds to see Ax improve upon the historical data, then examine the final results.

In [11]:
for _ in range(5):
    trials = client.get_next_trials(max_trials=1)
    for trial_index, parameters in trials.items():
        result = branin(parameters["x1"], parameters["x2"])
        client.complete_trial(
            trial_index=trial_index,
            raw_data={"branin": result},
        )

[INFO 02-19 16:21:12] ax.api.client: Generated new trial 16 with parameters {'x1': 10.0, 'x2': 3.051505} using GenerationNode MBM.
[INFO 02-19 16:21:12] ax.api.client: Trial 16 marked COMPLETED.
[INFO 02-19 16:21:13] ax.api.client: Generated new trial 17 with parameters {'x1': -1.478571, 'x2': 9.250075} using GenerationNode MBM.
[INFO 02-19 16:21:13] ax.api.client: Trial 17 marked COMPLETED.
[INFO 02-19 16:21:13] ax.api.client: Generated new trial 18 with parameters {'x1': 10.0, 'x2': 0.0} using GenerationNode MBM.
[INFO 02-19 16:21:13] ax.api.client: Trial 18 marked COMPLETED.
[INFO 02-19 16:21:13] ax.api.client: Generated new trial 19 with parameters {'x1': -2.930578, 'x2': 15.0} using GenerationNode MBM.
[INFO 02-19 16:21:13] ax.api.client: Trial 19 marked COMPLETED.
[INFO 02-19 16:21:13] ax.api.client: Generated new trial 20 with parameters {'x1': -3.225827, 'x2': 12.428191} using GenerationNode MBM.
[INFO 02-19 16:21:13] ax.api.client: Trial 20 marked COMPLETED.


In [12]:
best_parameters, prediction, trial_index, arm_name = (
    client.get_best_parameterization()
)
print(f"Best parameters: {best_parameters}")
print(f"Prediction (mean, sem): {prediction}")
print(f"From trial {trial_index} (arm '{arm_name}')")
print(f"\nKnown global minimum: ~0.398 at (x1, x2) in "
      "{(-pi, 12.275), (pi, 2.275), (9.425, 2.475)}")

Best parameters: {'x1': 10.0, 'x2': 3.051504770822107}
Prediction (mean, sem): {'branin': (np.float64(1.9996011891514804), np.float64(3.26333652262047))}
From trial 16 (arm '16_0')

Known global minimum: ~0.398 at (x1, x2) in {(-pi, 12.275), (pi, 2.275), (9.425, 2.475)}


## Step 9: Final Analysis

Let's compute a final suite of analyses to see how well the optimization performed after warm-starting from historical data. We include:
- **Cross-Validation**: Updated model fit assessment with all data
- **Utility Progression**: Shows how the best value improved — including through the historical warm-start phase and the Bayesian optimization phase
- **Modeled Arm Effects**: Model-based predictions for all arms with uncertainty estimates

In [ ]:
cards = client.compute_analyses(
    analyses=[
        CrossValidationPlot(),
        UtilityProgressionAnalysis(),
        ArmEffectsPlot(metric_name="branin", use_model_predictions=True),
    ],
    display=True,
)

## Conclusion

This tutorial demonstrated how to use Ax's `Client` to:
- **Attach historical trial data** from a DataFrame using `attach_trial` and `complete_trial`
- **Analyze the experiment** with cross-validation, utility progression, and arm effects plots
- **Generate new candidates** via Bayesian optimization, warm-started on historical observations
- **Visualize predicted arm effects** to understand model expectations before evaluating candidates

This workflow is applicable to any scenario where you have pre-existing evaluations and want to continue optimizing using Ax's Bayesian optimization engine.